# Enterprise Geochemical Database Management System
**Objective**

The objective of this project was to transform legacy USGS geochemical datasets stored in DBF format into a modern data management and reporting platform using Python, SQL Server, and Power BI.

**Dataset**

Source: United States Geological Survey (USGS) National Geochemical Database

The project integrates multiple geochemical datasets containing sample metadata, analytical measurements, laboratory methods, and QA/QC qualifiers.

## Phase 1: Data Ingestion and Initial Database Audit
This project uses three major geochemical databases obtained from the USGS National Geochemical Database (NGDB):

**GEODATA.DBF** (Metadata and Sample Information)

**MAJORS.DBF** (Major Element Geochemistry)

**XRF.DBF** (Trace Element Geochemistry)

The objective of this phase is to:
- Load and inspect the source databases
- Evaluate dataset dimensions
- Measure memory usage
- Verify data integrity
- Prepare the data for cleaning, QA/QC, and SQL Server migration

This workflow forms the foundation of an enterprise-scale geochemical database management system.


In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

from pathlib import Path
from dbfread import DBF


In [2]:
geo_path = r"D:\old_files\USGS\tblRockGeoData\GEODATA.DBF"

maj_path = r"D:\old_files\USGS\xtbMajorChem\MAJORS.DBF"

xrf_path = r"D:\old_files\USGS\xtbXrfChem\XRF.DBF"

##### Source File Size Assessment
Before loading the databases into memory, file sizes are evaluated.

This step is important because large geochemical datasets can exceed available RAM and cause performance issues during processing.

Understanding database size helps guide memory-efficient workflows and future SQL migration strategies.

In [3]:
for file in [geo_path, maj_path, xrf_path]:
    
    size_mb = Path(file).stat().st_size / 1024 / 1024 # returns the size of the file in bytes / 1024 twice to convert bytes into megabytes (MB)
    
    print(f"{Path(file).name:<15} {size_mb:,.2f} MB")

GEODATA.DBF     725.42 MB
MAJORS.DBF      321.28 MB
XRF.DBF         32.09 MB


This step provides a quick estimate of the storage requirements of the source databases before loading them into memory.

It is important to note that file size on disk is not the same as memory usage in Python. A DBF file may occupy significantly more memory once loaded into a pandas DataFrame. Therefore, evaluating file sizes helps anticipate potential memory constraints prior to data ingestion.

### Load USGS Source Databases
The three source databases are loaded into pandas DataFrames.

The DBF format is a legacy database format commonly used by geological surveys, laboratory systems, and GIS applications.

The latin1 encoding is used to avoid character decoding issues that may occur in older USGS databases.

Loading the databases into pandas allows us to profile, validate, clean, and prepare the datasets before transferring them into SQL Server

In [4]:
geo = pd.DataFrame(
    DBF(geo_path, encoding="latin1")
)

maj = pd.DataFrame(
    DBF(maj_path, encoding="latin1")
)

xrf = pd.DataFrame(
    DBF(xrf_path, encoding="latin1")
)

After loading the source databases, the first validation step is to examine their dimensions.

The shape attribute returns:

(number of rows, number of columns)

This provides an overview of database size and complexity before any profiling or cleaning operations are performed.

Understanding the number of records and variables is critical for estimating memory requirements, planning SQL Server schema design, and identifying potential performance bottlenecks.

In [5]:
print("GEODATA:", geo.shape)
print("MAJORS :", maj.shape)
print("XRF    :", xrf.shape)

GEODATA: (414304, 31)
MAJORS : (167937, 191)
XRF    : (38581, 83)


The row counts indicate that GEODATA serves as the primary sample inventory, while MAJORS and XRF represent analytical subsets. This structure naturally supports a relational database design in which GEODATA functions as the parent table and analytical datasets are linked through LAB_ID.

Before doing any cleaning, we need to answer a critical database question:

**Is LAB_ID unique in each table?**

In [6]:
print("GEODATA:", geo["LAB_ID"].is_unique)
print("MAJORS :", maj["LAB_ID"].is_unique)
print("XRF    :", xrf["LAB_ID"].is_unique)

GEODATA: True
MAJORS : True
XRF    : True


This result indicates that LAB_ID is a suitable candidate primary key for each source database and can be safely used for relational joins and future SQL Server schema design.

The successful validation also reduces the risk of row multiplication during database merges and supports referential integrity across analytical tables.

In [7]:
# Do all MAJORS records exist in GEODATA?
maj_in_geo = maj["LAB_ID"].isin(geo["LAB_ID"]).all()
# Do all XRF records exist in GEODATA?
xrf_in_geo = xrf["LAB_ID"].isin(geo["LAB_ID"]).all()

print("MAJORS -> GEODATA:", maj_in_geo)
print("XRF -> GEODATA:", xrf_in_geo)

MAJORS -> GEODATA: True
XRF -> GEODATA: True


- Every record in MAJORS has a matching LAB_ID in GEODATA.
- Every record in XRF has a matching LAB_ID in GEODATA.
- No orphan analytical records were detected.
- Referential integrity is already present in the source databases.

GEODATA.LAB_ID --> the parent key in the future SQL Server database.

MAJORS.LAB_ID  --> foreign key
XRF.LAB_ID     --> foreign key

Before any cleaning or merging, let's measure the ``actual RAM usage`` of the three DataFrames.

In [8]:
for name, df in {
    "GEODATA": geo,
    "MAJORS": maj,
    "XRF": xrf
}.items():

    mem_mb = (
        df.memory_usage(deep=True)
        .sum()
        / 1024**2
    )

    print(f"{name}: {mem_mb:.2f} MB")

GEODATA: 710.02 MB
MAJORS: 1029.54 MB
XRF: 103.33 MB


The combined memory footprint of the three databases is approximately 1.84 GB.

Although the source DBF files occupy only about 1.08 GB on disk, the in-memory representation requires substantially more memory due to DataFrame indexing, metadata storage, and Python object overhead.

The MAJORS table represents the largest memory consumer and will be the primary target for data reduction and optimization during later ETL stages.

### Inspect the structure of each table
This step provides a detailed understanding of the metadata, analytical, and sample information stored within the source databases and supports data cleaning and schema design decisions.

##### First geo

In [9]:
geo.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414304 entries, 0 to 414303
Data columns (total 31 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   LAB_ID      414304 non-null  object 
 1   JOB_ID      414304 non-null  object 
 2   SUBMITTER   414304 non-null  object 
 3   DATE_SUB    408666 non-null  object 
 4   FIELD_ID    414304 non-null  object 
 5   STATE       414304 non-null  object 
 6   COUNTRY     414304 non-null  object 
 7   DATUM       414304 non-null  object 
 8   SPHEROID    414304 non-null  object 
 9   LATITUDE    414304 non-null  float64
 10  LONGITUDE   414304 non-null  float64
 11  DEPTH       414304 non-null  object 
 12  LOCAT_DESC  414304 non-null  object 
 13  DATECOLLCT  414304 non-null  object 
 14  SAMPLE_SRC  414304 non-null  object 
 15  METHCOLLCT  414304 non-null  object 
 16  PRIMECLASS  414304 non-null  object 
 17  XNDRYCLASS  414304 non-null  object 
 18  SPEC_NAME   414304 non-null  object 
 19  AD

DATE_SUB is an object. Change the format:

In [10]:
geo["DATE_SUB"] = pd.to_datetime(
    geo["DATE_SUB"],
    errors="coerce"
)

### Missing Value Assessment

In [11]:
missing = (geo.isna().sum().sort_values(ascending=False))

print(missing)

missing_pct = (geo["DATE_SUB"].isna().mean() * 100)
print(f"Missing DATE_SUB: {missing_pct:.2f}%")

DATE_SUB      5638
JOB_ID           0
LAB_ID           0
SUBMITTER        0
FIELD_ID         0
STATE            0
COUNTRY          0
DATUM            0
SPHEROID         0
LATITUDE         0
LONGITUDE        0
DEPTH            0
LOCAT_DESC       0
DATECOLLCT       0
SAMPLE_SRC       0
METHCOLLCT       0
PRIMECLASS       0
XNDRYCLASS       0
SPEC_NAME        0
ADDL_ATTR        0
GEOL_AGE         0
STRATGRPHY       0
MINERALZTN       0
ALTERATION       0
STRUCT_SRC       0
DEP_ENVIRN       0
SOURCE_RK        0
METAMRPHSM       0
FACIES_GRD       0
PREP             0
MESH_SIZE        0
dtype: int64
Missing DATE_SUB: 1.36%


The metadata table demonstrates excellent data quality.

Out of 414,304 records, only the DATE_SUB field contains missing values.

Missing DATE_SUB records: 5,638

This represents approximately 1.36% of the dataset.

All other metadata fields contain complete records with no missing values detected.

These results indicate that GEODATA is suitable to serve as the parent table in the future SQL Server database and requires minimal preprocessing before database migration.

##### second Maj

In [12]:
maj.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167937 entries, 0 to 167936
Columns: 191 entries, LAB_ID to F_UNKQ
dtypes: float64(95), object(96)
memory usage: 244.7+ MB


In [13]:
maj.head(5)

,LAB_ID,SIO2_XRF,SIO2_XRFQ,SIO2_GRV,SIO2_GRVQ,SIO2_AES,SIO2_AESQ,SIO2_AA,SIO2_AAQ,SIO2_CLR,...,CL_MSC,CL_MSCQ,CL_UNK,CL_UNKQ,F_ELCTRD,F_ELCTRDQ,F_MSC,F_MSCQ,F_UNK,F_UNKQ
0,AAV245,NaN,,NaN,,NaN,,NaN,,NaN,...,NaN,,NaN,,0.32,,NaN,,NaN,
1,AAV247,NaN,,NaN,,NaN,,NaN,,NaN,...,NaN,,NaN,,0.60,,NaN,,NaN,
2,AAV248,NaN,,NaN,,NaN,,NaN,,NaN,...,NaN,,NaN,,0.56,,NaN,,NaN,
3,AAV249,NaN,,NaN,,NaN,,NaN,,NaN,...,NaN,,NaN,,0.27,,NaN,,NaN,
4,AAV251,NaN,,NaN,,NaN,,NaN,,NaN,...,NaN,,NaN,,0.46,,NaN,,NaN,


In [14]:
maj.columns.tolist()[:50]

['LAB_ID',
 'SIO2_XRF',
 'SIO2_XRFQ',
 'SIO2_GRV',
 'SIO2_GRVQ',
 'SIO2_AES',
 'SIO2_AESQ',
 'SIO2_AA',
 'SIO2_AAQ',
 'SIO2_CLR',
 'SIO2_CLRQ',
 'SIO2_UNK',
 'SIO2_UNKQ',
 'AL2O3_XRF',
 'AL2O3_XRFQ',
 'AL2O3_GRV',
 'AL2O3_GRVQ',
 'AL2O3_AES',
 'AL2O3_AESQ',
 'AL2O3_AA',
 'AL2O3_AAQ',
 'AL2O3_CLR',
 'AL2O3_CLRQ',
 'AL2O3_UNK',
 'AL2O3_UNKQ',
 'FE2O3_XRF',
 'FE2O3_XRFQ',
 'FE2O3_GRV',
 'FE2O3_GRVQ',
 'FE2O3_AES',
 'FE2O3_AESQ',
 'FE2O3_AA',
 'FE2O3_AAQ',
 'FE2O3_CLR',
 'FE2O3_CLRQ',
 'FE2O3_UNK',
 'FE2O3_UNKQ',
 'FEO_AA',
 'FEO_AAQ',
 'FEO_AES',
 'FEO_AESQ',
 'FEO_GRV',
 'FEO_GRVQ',
 'FEO_MSC',
 'FEO_MSCQ',
 'FEO_TTR',
 'FEO_TTRQ',
 'FEO_UNK',
 'FEO_UNKQ',
 'FETO3_XRF']

In [15]:
maj.iloc[:, :20].head()

,LAB_ID,SIO2_XRF,SIO2_XRFQ,SIO2_GRV,SIO2_GRVQ,SIO2_AES,SIO2_AESQ,SIO2_AA,SIO2_AAQ,SIO2_CLR,SIO2_CLRQ,SIO2_UNK,SIO2_UNKQ,AL2O3_XRF,AL2O3_XRFQ,AL2O3_GRV,AL2O3_GRVQ,AL2O3_AES,AL2O3_AESQ,AL2O3_AA
0,AAV245,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN
1,AAV247,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN
2,AAV248,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN
3,AAV249,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN
4,AAV251,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN,,NaN


**Which columns are mostly empty?**

In [16]:
maj_missing = (
    maj.isna()
       .mean()
       .mul(100)
       .sort_values(ascending=False)
)

maj_missing.head(30)

FETO3_MSC    99.997023
CO2_XRF      99.997023
FE2O3_AES    99.995236
FETO3_AES    99.987495
FEO_AES      99.976182
LOF_UNK      99.965463
S_ORG_MSC    99.963677
CL_CLR       99.961890
TIO2_AES     99.958913
MGO_AES      99.942836
K2O_AES      99.942836
CAO_AES      99.942836
AL2O3_AES    99.942836
MNO_AES      99.942836
P2O5_AES     99.942836
FETO3_CLR    99.929736
CO2_GRV      99.887458
NA2O_GRV     99.886862
TIO2_GRV     99.886862
P2O5_GRV     99.886862
CAO_GRV      99.886862
K2O_GRV      99.886862
SIO2_GRV     99.886862
AL2O3_GRV    99.886862
FE2O3_GRV    99.886862
FEO_GRV      99.886862
MGO_GRV      99.886862
MNO_GRV      99.886862
CL_XRF       99.833271
CL_MSC       99.816002
dtype: float64

Initial Sparsity Assessment
The MAJORS dataset contains a large number of highly sparse analytical fields.

Several analytical-method columns exhibit more than 99% missing values, indicating that those analytical techniques were applied to only a very small subset of samples.

Examples include:
- FETO3_MSC
- CO2_XRF
- FE2O3_AES
- FETO3_AES
  
The presence of extremely sparse columns suggests potential opportunities for database optimization, schema normalization, and storage reduction during later ETL stages.

At this stage, no columns are removed. The objective is to document data availability patterns before making any cleaning decisions.

In [17]:
maj_missing.describe()

count    191.000000
mean      44.714063
std       46.232092
min        0.000000
25%        0.000000
50%        0.000000
75%       97.369252
max       99.997023
dtype: float64

- At least 50% of the columns have no missing values at all.
- The dataset is not generally poor quality.
- The problem is concentrated in a subset of analytical-method columns.
- About 25% of the columns are extremely sparse (>97% missing).



In [18]:
# Identify columns that are 100% populated.
(maj_missing == 0).sum()

np.int64(96)

In [19]:
# The number of columns are essentially empty
(maj_missing > 99).sum()

np.int64(38)

The MAJORS dataset contains 191 analytical fields.

Field completeness analysis indicates:
- 96 columns are fully populated with no missing values.
- 38 columns contain more than 99% missing values.
  
These results suggest that the majority of the dataset remains suitable for analysis, while a subset of analytical-method fields may represent rare or historically limited laboratory procedures.


In [20]:
# The columns that is literally 100% empty
(maj_missing == 100).sum()

np.int64(0)

A review of column completeness found `no fields with 100% missing values`.

Although several analytical-method columns contain extremely high proportions of missing values, every field contains at least some valid analytical information.

This finding indicates that no columns can be automatically excluded solely because they are entirely empty.

Future optimization decisions will therefore be based on scientific relevance, analytical coverage, and database design considerations rather than simple missing-value thresholds.

In [21]:
maj.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
SIO2_XRF,76424.0,59.116939,13.782980,0.10,50.3000,59.400,69.1000,99.60
SIO2_GRV,190.0,50.515474,3.608184,44.76,49.3625,50.535,50.8400,64.41
SIO2_AES,841.0,14.928407,20.020951,0.10,4.2000,7.400,13.5000,93.80
SIO2_AA,3524.0,56.904083,13.276437,0.43,48.8000,52.800,68.1000,98.30
SIO2_CLR,4498.0,57.164595,13.800623,0.01,48.3000,58.500,67.9000,97.20
SIO2_UNK,29134.0,58.037901,18.388102,0.01,49.3700,59.900,70.7000,100.00
AL2O3_XRF,76424.0,13.924687,4.579001,0.01,12.7000,14.900,16.6000,58.00
AL2O3_GRV,190.0,13.033789,1.873855,8.11,12.5525,13.520,13.8050,18.08
AL2O3_AES,96.0,12.109896,6.637089,0.21,7.4300,13.750,16.9000,32.90
AL2O3_AA,3524.0,13.923468,3.942020,0.02,13.0000,14.200,15.7000,37.80


In [22]:
maj.count().sort_values(ascending=False).head(30)

LAB_ID        167937
SIO2_XRFQ     167937
SIO2_GRVQ     167937
SIO2_AESQ     167937
AL2O3_XRFQ    167937
SIO2_CLRQ     167937
SIO2_AAQ      167937
SIO2_UNKQ     167937
FE2O3_XRFQ    167937
FE2O3_AESQ    167937
FE2O3_GRVQ    167937
AL2O3_AAQ     167937
AL2O3_CLRQ    167937
AL2O3_AESQ    167937
AL2O3_GRVQ    167937
AL2O3_UNKQ    167937
FETO3_XRFQ    167937
FETO3_CLRQ    167937
FETO3_AESQ    167937
MGO_XRFQ      167937
MGO_GRVQ      167937
FETO3_UNKQ    167937
FETO3_MSCQ    167937
FEO_AESQ      167937
FEO_GRVQ      167937
FEO_TTRQ      167937
FEO_MSCQ      167937
FE2O3_UNKQ    167937
FEO_AAQ       167937
FE2O3_CLRQ    167937
dtype: int64

In [23]:
maj["SIO2_XRFQ"].value_counts(dropna=False).head(20)

SIO2_XRFQ
     167909
L        16
>         5
<         4
N         2
G         1
Name: count, dtype: int64

##### Analytical Qualifier Assessment
The analytical qualifier fields (Q columns) were evaluated to determine their contribution to database content.

Investigation of representative qualifier fields showed that the vast majority of records contain blank values, while a very small number of samples contain analytical flags such as:
- <
- L
- N
- G
  
These flags likely represent `laboratory reporting qualifiers`, `detection-limit indicators`, or `analytical quality annotations`.

Although qualifier columns contain relatively little information compared to analytical value fields, they remain scientifically important and should be preserved within the database design for traceability and QA/QC purposes.

##### Quantify the value columns

In [24]:
value_cols = [c for c in maj.columns if not c.endswith("Q")]

len(value_cols)

96

### MAJORS Schema Assessment
The MAJORS database contains 191 columns.

Field classification identified:
- 96 analytical value fields
- 95 analytical qualifier fields
  
The structure follows a consistent pattern in which analytical measurements are paired with corresponding qualifier fields.

Examples:
- SIO2_XRF / SIO2_XRFQ
- AL2O3_XRF / AL2O3_XRFQ
- FE2O3_XRF / FE2O3_XRFQ
  
This design allows the database to retain both analytical results and laboratory reporting qualifiers for quality control and traceability purposes.

The field organization will be considered during future SQL Server schema design and normalization activities.

In [25]:
value_missing = (
    maj[value_cols]
    .isna()
    .mean()
    .mul(100)
)

value_missing.describe()

count    96.000000
mean     88.962354
std      17.232076
min       0.000000
25%      83.024735
50%      97.369252
75%      99.886862
max      99.997023
dtype: float64

The high proportion of missing values observed within MAJORS does not indicate poor data quality.

Instead, the sparsity reflects the historical use of multiple analytical methods across different sample populations.

Most samples were analyzed using a limited subset of available laboratory methods, resulting in a highly sparse wide-table structure.

This finding highlights the importance of database normalization and careful schema design when migrating legacy geochemical databases into modern relational database systems.

## Phase 2: Database Engineering and Logical Data Model Design

The original USGS databases were developed as legacy DBF tables designed for data storage and laboratory reporting.

While suitable for data capture, the original structure is not optimized for modern relational database management, analytical reporting, or business intelligence applications.

The objective of this phase is to redesign the source databases into a scalable SQL Server architecture that supports:

- Data integrity
- Relational database management
- QA/QC workflows
- Analytical reporting
- Power BI integration
- Future data expansion
  
Rather than directly migrating the original DBF files into SQL Server, the database will be restructured using modern database engineering principles.

The original database structure contains analytical results stored as wide tables.

Example:

`LAB_ID` `SIO2_XRF` `SIO2_XRFQ` `SIO2_AA` `SIO2_AAQ` `AL2O3_XRF` `AL2O3_XRFQ`

This structure creates:

- High sparsity
- Difficult reporting
- Limited scalability
- Complex Power BI models
  
A normalized database design will improve maintainability, reporting performance, and future analytical capabilities.

### Relational Database Architecture Design
Rather than directly transferring the legacy DBF tables into SQL Server, the database will be redesigned using relational database principles.

The original USGS structure stores analytical results in wide tables containing hundreds of columns. While suitable for historical data storage, this design is difficult to maintain, scale, query, and visualize.

The redesigned architecture separates `samples`, `analytical results`, `methods`, `elements`, and `qualifiers` into individual relational entities.

This approach provides:

- Improved scalability
- Reduced redundancy
- Better SQL performance
- Simplified Power BI integration
- Long-term maintainability
- Support for additional analytical methods

##### Proposed SQL Server Schema
**Samples**-->LAB_ID (PK)

**Elements**-->ElementID (PK)

**Methods**-->MethodID (PK)

**ChemistryResults**-->ResultID (PK) LAB_ID (FK) ElementID (FK) MethodID (FK) Value QualifierID (FK)

**Qualifiers**-->QualifierID (PK) QualifierCode Description


##### Separation of Analytical Values and Qualifiers
The MAJORS table contains both geochemical measurements and laboratory qualifier fields. Columns ending with "Q" represent analytical qualifiers, while the remaining columns contain actual analytical results.

To support database `normalization` and `relational schema design`, analytical value fields were separated from qualifier fields. This distinction will be used to build the Elements, Methods, and ChemistryResults tables within the SQL Server database.

In [26]:
value_cols = [c for c in maj.columns if not c.endswith("Q")]

value_cols[:30]

['LAB_ID',
 'SIO2_XRF',
 'SIO2_GRV',
 'SIO2_AES',
 'SIO2_AA',
 'SIO2_CLR',
 'SIO2_UNK',
 'AL2O3_XRF',
 'AL2O3_GRV',
 'AL2O3_AES',
 'AL2O3_AA',
 'AL2O3_CLR',
 'AL2O3_UNK',
 'FE2O3_XRF',
 'FE2O3_GRV',
 'FE2O3_AES',
 'FE2O3_AA',
 'FE2O3_CLR',
 'FE2O3_UNK',
 'FEO_AA',
 'FEO_AES',
 'FEO_GRV',
 'FEO_MSC',
 'FEO_TTR',
 'FEO_UNK',
 'FETO3_XRF',
 'FETO3_AES',
 'FETO3_CLR',
 'FETO3_MSC',
 'FETO3_UNK']

##### Identification of Elements and Analytical Methods
The original MAJORS table stores geochemical measurements as compound column names that combine both the element and the analytical method.

Examples:
- SIO2_XRF
- AL2O3_XRF
- FE2O3_AA

For database normalization, these compound field names must be separated into individual entities.

This step extracts:
- Element names (SiO2, Al2O3, Fe2O3, etc.)
- Analytical methods (XRF, AA, AES, GRV, etc.)
  
The resulting information will populate the Elements and Methods dimension tables within the relational SQL Server database.

In [27]:
value_cols = [
    c for c in maj.columns
    if not c.endswith("Q")
    and c != "LAB_ID"
]

split_cols = [c.rsplit("_", 1) for c in value_cols]

elements = sorted(set(x[0] for x in split_cols))
methods = sorted(set(x[1] for x in split_cols))

print("Number of Elements:", len(elements))
print("Number of Methods :", len(methods))

print("\nMethods:")
print(methods)

Number of Elements: 27
Number of Methods : 11

Methods:
['AA', 'AES', 'CLR', 'CMB', 'DFF', 'ELCTRD', 'GRV', 'MSC', 'TTR', 'UNK', 'XRF']


##### Extraction of Elements and Analytical Methods
The previous analysis identified 96 analytical value fields within the MAJORS table. These fields follow a naming convention that combines a chemical constituent with an analytical method.

Examples:
- SIO2_XRF
- AL2O3_AES
- FE2O3_AA

To support database normalization, each column name was decomposed into two separate entities:
- Element
- Analytical Method

This process identified:
- 27 unique elements
- 11 analytical methods
  
These entities will become independent dimension tables within the SQL Server database, allowing analytical results to be stored in a normalized and scalable structure.

In [28]:
value_cols = [
    c for c in maj.columns
    if not c.endswith("Q")
    and c != "LAB_ID"
]

split_cols = [c.rsplit("_", 1) for c in value_cols]

elements = sorted(
    set(x[0] for x in split_cols)
)

methods = sorted(
    set(x[1] for x in split_cols)
)

print("Unique Elements:", len(elements))
print("Unique Methods :", len(methods))

Unique Elements: 27
Unique Methods : 11


In [29]:
methods_df = pd.DataFrame(
    {
        "MethodID": range(1, len(methods)+1),
        "MethodName": methods
    }
)

methods_df

,MethodID,MethodName
0,1,AA
1,2,AES
2,3,CLR
3,4,CMB
4,5,DFF
5,6,ELCTRD
6,7,GRV
7,8,MSC
8,9,TTR
9,10,UNK


##### Methods Dimension Table
The normalization process identified 11 unique analytical methods within the MAJORS database.

Rather than storing analytical methods within column names, a dedicated Methods dimension table is created.

This approach improves database normalization, reduces redundancy, and allows future analytical methods to be added without modifying the database schema.

The Methods table will be linked to analytical results through a foreign key relationship in the ChemistryResults table.

##### Elements Dimension Table
The MAJORS database contains multiple chemical constituents measured by various laboratory methods.

To support a relational database structure, unique chemical constituents are extracted and stored in a dedicated Elements dimension table.

This design separates chemical constituents from laboratory methods and allows analytical results to be stored in a normalized format.

The Elements table will become a parent entity for all geochemical measurements stored in the ChemistryResults table.

In [30]:
elements_df = pd.DataFrame(
    {
        "ElementID": range(1, len(elements) + 1),
        "ElementName": elements
    }
)

elements_df

,ElementID,ElementName
0,1,AL2O3
1,2,CAO
2,3,CL
3,4,CO2
4,5,C_CRB
5,6,C_ORG
6,7,C_TOT
7,8,F
8,9,FE2O3
9,10,FEO


##### Elements Dimension Table
The normalization process identified 27 unique chemical constituents within the MAJORS database.

Rather than embedding chemical constituents within analytical column names, a dedicated Elements dimension table is created.

This design improves data consistency, reduces redundancy, and supports scalable storage of analytical results. The Elements table serves as a parent entity for all future geochemical measurements stored within the ChemistryResults fact table.

Examples include:

SiO2 | Al2O3 | Fe2O3 | MgO | TiO2

By separating elements from analytical methods, the database becomes more flexible and can accommodate future analytical programs without requiring schema modifications.

##### Transformation to Long Format
The original MAJORS database stores analytical results in a wide-table structure in which each analytical result is represented by a separate column.

To support relational database design, the dataset is transformed into a long format structure.

This transformation converts analytical columns into individual records, allowing each analytical result to be stored as a separate database row.

The resulting structure forms the foundation of the ChemistryResults fact table and supports database normalization, SQL querying, and Power BI reporting.

In [31]:
value_cols = [
    c for c in maj.columns
    if not c.endswith("Q")
    and c != "LAB_ID"
]
# The melt() method converts a DataFrame from wide format to long format by turning columns into rows.
maj_long = maj.melt(
    id_vars="LAB_ID",
    value_vars=value_cols,
    var_name="Element_Method",
    value_name="Value"
)

maj_long.head()

,LAB_ID,Element_Method,Value
0,AAV245,SIO2_XRF,NaN
1,AAV247,SIO2_XRF,NaN
2,AAV248,SIO2_XRF,NaN
3,AAV249,SIO2_XRF,NaN
4,AAV251,SIO2_XRF,NaN


##### Removal of Empty Analytical Records
The transformation to long format generates a record for every possible analytical field associated with each sample.

Because many laboratory methods were not applied to every sample, the resulting dataset contains a large number of null analytical records.

For database implementation, only valid analytical measurements will be retained. `Removing null analytical record` significantly reduces dataset size and creates an efficient fact table suitable for SQL Server storage and analytical reporting.

In [32]:
maj_long = maj_long.dropna(subset=["Value"])

maj_long.head()

,LAB_ID,Element_Method,Value
5557,C100666,SIO2_XRF,47.8
5558,C100667,SIO2_XRF,47.7
5559,C100668,SIO2_XRF,49.9
5560,C100669,SIO2_XRF,47.1
5561,C100670,SIO2_XRF,44.6


The resulting dataset forms the foundation of the ChemistryResults fact table within the proposed relational database architecture.

In [33]:
maj_long.shape

(1611547, 3)

After transforming the database into a normalized long-format structure and removing records with null analytical values, a total of 1,611,547 valid laboratory measurements were retained.

This transformation significantly reduces database sparsity and provides the foundation for the ChemistryResults fact table.

The resulting structure allows each analytical measurement to be stored as an individual record linked to:
- Samples
- Elements
- Analytical Methods
- Qualifiers

This design follows relational database best practices and supports efficient SQL querying, QA/QC workflows, and Power BI reporting.


##### Separation of Elements and Analytical Methods
To support database normalization, the combined analytical field names (e.g., SIO2_XRF) were separated into individual Element and Method components.

This transformation allows analytical results to be linked to dedicated Elements and Methods dimension tables through relational database relationships.

In [34]:
maj_long[["Element", "Method"]] = (
    maj_long["Element_Method"]
    .str.rsplit("_", n=1, expand=True)
)

maj_long.head()

,LAB_ID,Element_Method,Value,Element,Method
5557,C100666,SIO2_XRF,47.8,SIO2,XRF
5558,C100667,SIO2_XRF,47.7,SIO2,XRF
5559,C100668,SIO2_XRF,49.9,SIO2,XRF
5560,C100669,SIO2_XRF,47.1,SIO2,XRF
5561,C100670,SIO2_XRF,44.6,SIO2,XRF


The analytical field names were successfully separated into individual Element and Method components. This transformation converts the original wide-table structure into a normalized format that can be linked to the Elements and Methods dimension tables.

example:

SIO2_XRF --> Element = SIO2, Method = XRF

This structure forms the basis of the ChemistryResults fact table within the relational database design.

##### Creation of Foreign Key References

In [35]:
# Convert Element and Method names into ElementID and MethodID for a relational SQL Server design.
element_lookup = dict(
    zip(elements_df["ElementName"],
        elements_df["ElementID"])
)

method_lookup = dict(
    zip(methods_df["MethodName"],
        methods_df["MethodID"])
)

maj_long["ElementID"] = (
    maj_long["Element"]
    .map(element_lookup)
)

maj_long["MethodID"] = (
    maj_long["Method"]
    .map(method_lookup)
)

maj_long.head()


,LAB_ID,Element_Method,Value,Element,Method,ElementID,MethodID
5557,C100666,SIO2_XRF,47.8,SIO2,XRF,22,11
5558,C100667,SIO2_XRF,47.7,SIO2,XRF,22,11
5559,C100668,SIO2_XRF,49.9,SIO2,XRF,22,11
5560,C100669,SIO2_XRF,47.1,SIO2,XRF,22,11
5561,C100670,SIO2_XRF,44.6,SIO2,XRF,22,11


Element and Method names were successfully mapped to their corresponding IDs. This creates foreign key relationships between the ChemistryResults table and the Elements and Methods dimension tables.

For example:

SIO2 --> ElementID 22 XRF --> MethodID 11

The database now follows a relational structure that supports normalization and efficient SQL querying.

Now that we have foreign keys, we can create the first version of the actual ChemistryResults table by keeping only the columns required for SQL Server.

In [36]:
# Create the normalized fact table that will later be imported into SQL Server.
chemistry_results = maj_long[
    [
        "LAB_ID",
        "ElementID",
        "MethodID",
        "Value"
    ]
].copy()

chemistry_results.head()

,LAB_ID,ElementID,MethodID,Value
5557,C100666,22,11,47.8
5558,C100667,22,11,47.7
5559,C100668,22,11,49.9
5560,C100669,22,11,47.1
5561,C100670,22,11,44.6


A normalized ChemistryResults table was created by retaining only the fields required for relational database storage.

Each record now represents a single laboratory measurement linked to a sample, analytical method, and chemical constituent through foreign key relationships.

This structure forms the core fact table of the proposed SQL Server database.

**Create a surrogate key** (`ResultID`) that will serve as the primary key of the ChemistryResults table.

In [37]:
chemistry_results.insert(
    0,
    "ResultID",
    range(1, len(chemistry_results) + 1)
)

chemistry_results.head()

,ResultID,LAB_ID,ElementID,MethodID,Value
5557,1,C100666,22,11,47.8
5558,2,C100667,22,11,47.7
5559,3,C100668,22,11,49.9
5560,4,C100669,22,11,47.1
5561,5,C100670,22,11,44.6


A unique ResultID was added to each analytical record to serve as the primary key of the ChemistryResults table.

This approach follows relational database best practices and ensures that every laboratory measurement can be uniquely identified within SQL Server.

**Verify every analytical record is uniquely identified** before loading it into SQL Server. Validate the new ResultID can safely function as the primary key of the ChemistryResults table.

In [38]:
chemistry_results["ResultID"].is_unique

True

This confirms that ResultID can serve as the primary key of the ChemistryResults table, while LAB_ID, ElementID, and MethodID function as foreign keys linking analytical measurements to their respective dimension tables.

**Determine how many analytical measurements will be stored in the SQL Server ChemistryResults table.**

In [39]:
len(chemistry_results)

1611547

After normalization and removal of null analytical records, the ChemistryResults table contains 1,611,547 valid laboratory measurements.

This confirms that the original wide-table structure was successfully transformed into a relational fact table suitable for SQL Server implementation, QA/QC workflows, and Power BI reporting.

**Validate the foreign key relationship between ChemistryResults and Samples.**

In [40]:
chemistry_results["LAB_ID"].isin(
    geo["LAB_ID"]
).all()

np.True_

All analytical measurements were successfully matched to valid sample records using LAB_ID.

This confirms referential integrity between the ChemistryResults fact table and the Samples table, ensuring that every analytical result can be traced back to its source sample.


**Evaluate the storage footprint of the normalized ChemistryResults table before SQL Server implementation.**

In [41]:
chemistry_results.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1611547 entries, 5557 to 15948521
Data columns (total 5 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   ResultID   1611547 non-null  int64  
 1   LAB_ID     1611547 non-null  object 
 2   ElementID  1611547 non-null  int64  
 3   MethodID   1611547 non-null  int64  
 4   Value      1611547 non-null  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 73.8+ MB


The normalized ChemistryResults table contains 1,611,547 analytical measurements and occupies approximately 74 MB of memory.

Despite representing more than 1.6 million laboratory results, the normalized structure remains compact and efficient, demonstrating the benefits of database normalization over the original wide-table design.

The resulting table is suitable for SQL Server implementation and large-scale analytical reporting.

##### Creation of the Samples Dimension Table
We now need to create the Samples dimension table from GEODATA. This table will become the parent table of the entire SQL Server database and will be linked to all analytical results through LAB_ID.

**Select only the business-relevant metadata fields that should be stored in the SQL Server Samples table.**

In [42]:
samples = geo[
    [
        "LAB_ID",
        "STATE",
        "COUNTRY",
        "LATITUDE",
        "LONGITUDE",
        "DATE_SUB",
        "PRIMECLASS",
        "XNDRYCLASS",
        "SPEC_NAME",
        "GEOL_AGE",
        "STRATGRPHY",
        "MINERALZTN",
        "ALTERATION"
    ]
].copy()

samples.head()

,LAB_ID,STATE,COUNTRY,LATITUDE,LONGITUDE,DATE_SUB,PRIMECLASS,XNDRYCLASS,SPEC_NAME,GEOL_AGE,STRATGRPHY,MINERALZTN,ALTERATION
0,AAM051,,,34.46889,-83.97139,1967-07-03,rock,,,,,,
1,AAM051A,,,34.46889,-83.97139,1967-07-03,rock,,,,,,
2,AAM052,,,34.46944,-83.97194,1967-07-03,rock,,,,,,
3,AAM052A,,,34.46944,-83.97194,1967-07-03,rock,,,,,,
4,AAM053,,,34.47000,-83.97361,1967-07-03,rock,,,,,,


A dedicated Samples dimension table was created from the GEODATA database by selecting the metadata fields required for geological interpretation, spatial analysis, and reporting.

The Samples table serves as the parent entity of the relational database and provides the link between sample metadata and laboratory analytical results through LAB_ID.

Now we need to verify that the Samples table can serve as the parent table of the entire database.

**Validate that LAB_ID remains unique after selecting the business-relevant fields.**

In [43]:
samples["LAB_ID"].is_unique

True

This confirms that LAB_ID can serve as the primary key of the Samples table and as the parent key linking sample metadata to analytical results throughout the database.

**Assess the storage footprint of the Samples dimension table and document its readiness for SQL Server implementation.**

Check the size of the Samples table we created.

In [44]:
samples.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414304 entries, 0 to 414303
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   LAB_ID      414304 non-null  object        
 1   STATE       414304 non-null  object        
 2   COUNTRY     414304 non-null  object        
 3   LATITUDE    414304 non-null  float64       
 4   LONGITUDE   414304 non-null  float64       
 5   DATE_SUB    408666 non-null  datetime64[ns]
 6   PRIMECLASS  414304 non-null  object        
 7   XNDRYCLASS  414304 non-null  object        
 8   SPEC_NAME   414304 non-null  object        
 9   GEOL_AGE    414304 non-null  object        
 10  STRATGRPHY  414304 non-null  object        
 11  MINERALZTN  414304 non-null  object        
 12  ALTERATION  414304 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(10)
memory usage: 41.1+ MB


The Samples table contains 414,304 unique sample records and occupies approximately 41 MB of memory.

The table includes spatial coordinates, geological classifications, lithological attributes, and sample metadata required for reporting, spatial analysis, and database integration.

The resulting structure is suitable for implementation as the parent dimension table within the SQL Server database.

**Qualifiers**

Identify all qualifier codes that will populate the future Qualifiers dimension table.

In [45]:
qualifier_cols = [
    c for c in maj.columns
    if c.endswith("Q")
]

qualifiers = pd.unique(
    pd.concat(
        [maj[col] for col in qualifier_cols]
    )
)

qualifiers = sorted(
    [q for q in qualifiers if pd.notna(q) and str(q).strip() != ""]
)

print(qualifiers)
print("\nNumber of Qualifiers:", len(qualifiers))

['<', '>', 'G', 'L', 'N']

Number of Qualifiers: 5


Five unique laboratory qualifier codes were identified within the analytical database.

These qualifier codes represent laboratory reporting annotations and QA/QC indicators associated with analytical results.

The Qualifiers table will support traceability and preserve analytical reporting information within the normalized database structure.

**Create the final dimension table of the model.**

Build the Qualifiers dimension table that will be linked to analytical results through foreign keys.

In [46]:
qualifiers_df = pd.DataFrame(
    {
        "QualifierID": range(1, len(qualifiers) + 1),
        "QualifierCode": qualifiers
    }
)

qualifiers_df

,QualifierID,QualifierCode
0,1,<
1,2,>
2,3,G
3,4,L
4,5,N


A dedicated Qualifiers dimension table was created to store laboratory reporting and QA/QC codes identified within the analytical database.

The table contains five unique qualifier codes that can be referenced by analytical results while preserving data integrity and reducing redundancy.

Together with the Samples, Elements, Methods, and ChemistryResults tables, the Qualifiers table completes the core relational database architecture.

##### Integration of Analytical Qualifiers
The original `MAJORS` database contains **laboratory qualifier fields (Q columns)** that store QA/QC and reporting flags associated with analytical measurements.

To complete the relational database design, qualifier values will be extracted, transformed into a normalized structure, and linked to analytical results through the Qualifiers dimension table.

This step ensures that analytical measurements retain their associated laboratory reporting information and supports full traceability within the database.

In [51]:
qualifier_cols = [
    c for c in maj.columns
    if c.endswith("Q")
]

qual_long = maj.melt(
    id_vars="LAB_ID",
    value_vars=qualifier_cols,
    var_name="Element_Method_Q",
    value_name="QualifierCode"
)

qual_long = qual_long.dropna(
    subset=["QualifierCode"]
)

qual_long = qual_long[
    qual_long["QualifierCode"].astype(str).str.strip() != ""
]

qual_long.head()

,LAB_ID,Element_Method_Q,QualifierCode
5609,C100865,SIO2_XRFQ,<
5620,C100876,SIO2_XRFQ,<
10081,C173704,SIO2_XRFQ,>
10093,C173716,SIO2_XRFQ,>
10095,C173718,SIO2_XRFQ,>


Qualifier records were successfully extracted from the original QA/QC fields and transformed into a normalized structure.

Each qualifier is now associated with a sample and analytical measurement, creating the foundation for linking the Qualifiers dimension table to the ChemistryResults fact table.

**Convert qualifier field names** into **Element and Method components**, exactly like we did for analytical values.

In [52]:
qual_long[["Element", "MethodQ"]] = (
    qual_long["Element_Method_Q"]
    .str.replace("Q", "", regex=False)
    .str.rsplit("_", n=1, expand=True)
)

qual_long.head()

,LAB_ID,Element_Method_Q,QualifierCode,Element,MethodQ
5609,C100865,SIO2_XRFQ,<,SIO2,XRF
5620,C100876,SIO2_XRFQ,<,SIO2,XRF
10081,C173704,SIO2_XRFQ,>,SIO2,XRF
10093,C173716,SIO2_XRFQ,>,SIO2,XRF
10095,C173718,SIO2_XRFQ,>,SIO2,XRF


The qualifier records were successfully transformed into the same relational structure used for analytical results.

Qualifier information is now associated with individual samples, elements, and analytical methods, making it possible to link the Qualifiers dimension table directly to the ChemistryResults fact table through a foreign key relationship

**Map qualifier codes (, >, L, N, G) to their corresponding QualifierID.** Create the foreign key needed to connect Qualifiers to ChemistryResults.


In [53]:
qualifier_lookup = dict(
    zip(
        qualifiers_df["QualifierCode"],
        qualifiers_df["QualifierID"]
    )
)

qual_long["QualifierID"] = (
    qual_long["QualifierCode"]
    .map(qualifier_lookup)
)

qual_long.head()

,LAB_ID,Element_Method_Q,QualifierCode,Element,MethodQ,QualifierID
5609,C100865,SIO2_XRFQ,<,SIO2,XRF,1
5620,C100876,SIO2_XRFQ,<,SIO2,XRF,1
10081,C173704,SIO2_XRFQ,>,SIO2,XRF,2
10093,C173716,SIO2_XRFQ,>,SIO2,XRF,2
10095,C173718,SIO2_XRFQ,>,SIO2,XRF,2


Qualifier codes were successfully mapped to their corresponding QualifierID values.

This transformation establishes the foreign key relationship required to connect QA/QC and laboratory reporting information from the Qualifiers table to analytical measurements stored in the ChemistryResults fact table.

**Join qualifier records to analytical records so that each analytical result can carry its associated QualifierID.**

In [54]:
qual_long["ElementID"] = (
    qual_long["Element"]
    .map(element_lookup)
)

qual_long["MethodID"] = (
    qual_long["MethodQ"]
    .map(method_lookup)
)

qual_long.head()

,LAB_ID,Element_Method_Q,QualifierCode,Element,MethodQ,QualifierID,ElementID,MethodID
5609,C100865,SIO2_XRFQ,<,SIO2,XRF,1,22,11
5620,C100876,SIO2_XRFQ,<,SIO2,XRF,1,22,11
10081,C173704,SIO2_XRFQ,>,SIO2,XRF,2,22,11
10093,C173716,SIO2_XRFQ,>,SIO2,XRF,2,22,11
10095,C173718,SIO2_XRFQ,>,SIO2,XRF,2,22,11


Qualifier records were successfully linked to both Elements and Methods through foreign keys.

The normalized qualifier structure now contains:
- LAB_ID
- ElementID
- MethodID
- QualifierID

This structure enables the integration of QA/QC qualifiers into the ChemistryResults fact table and completes the foundation of the five-table relational database model.

**Verify how many analytical records actually have qualifiers before modifying the database design.** Determine the number of analytical measurements that require a QualifierID.

In [55]:
len(qual_long)

97262

A total of 97,262 analytical records contain laboratory qualifier information.

This confirms that qualifier data is actively used within the database and supports the inclusion of a QualifierID foreign key in the final ChemistryResults design.

**Match analytical records with their corresponding qualifier records.** Create the final 5-table model by attaching QualifierID to ChemistryResults.

In [56]:
chemistry_results_q = chemistry_results.merge(
    qual_long[
        [
            "LAB_ID",
            "ElementID",
            "MethodID",
            "QualifierID"
        ]
    ],
    on=[
        "LAB_ID",
        "ElementID",
        "MethodID"
    ],
    how="left"
)

chemistry_results_q.head()

,ResultID,LAB_ID,ElementID,MethodID,Value,QualifierID
0,1,C100666,22,11,47.8,NaN
1,2,C100667,22,11,47.7,NaN
2,3,C100668,22,11,49.9,NaN
3,4,C100669,22,11,47.1,NaN
4,5,C100670,22,11,44.6,NaN


Qualifier information was successfully linked to analytical measurements using LAB_ID, ElementID, and MethodID.

Records without qualifier information retain a null QualifierID value, while records containing laboratory reporting flags are assigned the corresponding QualifierID.

This completes the five-table relational database design and establishes the relationship between ChemistryResults and Qualifiers.

**Check how many analytical records received a valid QualifierID.** Verify that the Qualifiers table is actually connected to ChemistryResults.

In [57]:
chemistry_results_q["QualifierID"].notna().sum()

np.int64(97262)

##### Phase 2 Summary
The original USGS geochemical databases were redesigned using relational database principles.

The legacy wide-table structure was normalized into a scalable SQL Server architecture consisting of:
- Samples
- Elements
- Methods
- Qualifiers
- ChemistryResults

A total of 1,611,547 valid laboratory measurements were transformed into a normalized fact table, with 97,262 records retaining QA/QC qualifier information through the Qualifiers dimension.

The resulting design reduces redundancy, improves scalability, preserves referential integrity, and provides a foundation for SQL Server implementation, QA/QC workflows, and Power BI reporting.

##### Dimension Table Validation

In [47]:
print(
    chemistry_results["ElementID"].isna().sum()
)

print(
    chemistry_results["MethodID"].isna().sum()
)

0
0


All analytical records were successfully linked to valid ElementID and MethodID values.

This confirms that the ChemistryResults fact table can be loaded into SQL Server while maintaining referential integrity with the Elements and Methods dimension tables.

In [48]:
samples["LAB_ID"].str.len().describe()

count    414304.000000
mean          6.561428
std           0.500986
min           5.000000
25%           6.000000
50%           7.000000
75%           7.000000
max           8.000000
Name: LAB_ID, dtype: float64

The maximum identifier length was found to be 8 characters.

Based on this assessment, LAB_ID will be implemented as VARCHAR(8), minimizing storage requirements while preserving all source identifiers.

### loading data
Verify the connection from Python to SQL Server before loading the first dimension table.

In [50]:
from sqlalchemy import create_engine

server = "BehrouzHP"
database = "USGS_Geochemistry"

engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

with engine.connect() as conn:
    print("Connected Successfully")

Connected Successfully


The SQL Server schema was successfully created and contains all core tables required by the relational database design:
- Samples
- Elements
- Methods
- Qualifiers
- ChemistryResults
This confirms that the normalized database architecture has been successfully implemented and is ready for data loading.

### Data Migration to SQL Server

In [59]:
chemistry_results_q.to_sql(
    "ChemistryResults",
    engine,
    schema="geo",
    if_exists="append",
    index=False,
    chunksize=1000
)

486720

During the migration process, approximately 486,720 records had been loaded when progress monitoring was performed.

The ChemistryResults fact table was successfully loaded into SQL Server using a chunked transfer strategy to accommodate the large dataset size.

A total of 1,611,547 analytical measurements were migrated while maintaining all primary key and foreign key relationships.

This completed the migration of the normalized geochemical database from Python into SQL Server and established the foundation for SQL-based analysis, reporting, and Power BI integration.